# Mumbai House Price Dataset

This notebook cleans and prepares a Mumbai house-price dataset for further analysis. Every code cell has a Markdown explanation immediately above it.

## 1. Load and explore the dataset

### 1. Import libraries

We will import the libraries needed to work with tabular data and create visualisations later in the analysis.

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

### 2. Load the raw dataset

We will read the CSV file into a pandas DataFrame called df and check its number of rows and columns.

In [2]:
df = pd.read_csv('Mumbai_Dataset_Unclean.csv')
display(df.shape)

(6597, 18)

### 3. Preview the data

We will display the first five records to understand the available columns and the format of the values.

In [3]:
display(df.head(5))

,price,sqrt,location,bhk,New/Resale,Gymnasium,Lift Available,CarParking,Maintenance Staff,24x7 Security,Children's Play Area,Clubhouse,Intercom,Landscaped Gardens,Indoor Games,Gas Connection,Jogging Track,Swimming Pool
0,NaN,1498.0,Kharghar,3.0,0.0,1,No,1,0,Yes,1,1,0,NaN,0,0,1,1
1,125000000.0,2904.0,WORLI,4.0,NaN,1,1,1,1,1,NaN,NaN,1,1,1,1,1,1
2,43000000.0,2195.0,Goregaon East,3.0,0.0,1,1,1,No,1,1,1,1,1,Yes,0,1,1
3,4500000.0,738.0,Taloja,1.0,0.0,1,1,1,No,1,1,1,1,1,1,NaN,1,1
4,6000000.0,650.0,Mira Road East,2.0,0.0,0,1,0,NaN,1,0,0,0,No,No,0,0,0


### 4. Review summary statistics

We will generate descriptive statistics for the numeric columns to understand their ranges and distributions.

In [4]:
display(df.describe())

,price,sqrt,bhk,New/Resale
count,6.062000e+03,6064.000000,6076.000000,6067.000000
mean,1.513401e+07,1011.533641,1.909151,0.343992
std,1.977278e+07,568.114484,0.868076,0.475078
min,2.000000e+06,200.000000,1.000000,0.000000
25%,5.300000e+06,650.000000,1.000000,0.000000
50%,9.500000e+06,910.000000,2.000000,0.000000
75%,1.750000e+07,1185.500000,2.000000,1.000000
max,4.000000e+08,8511.000000,7.000000,1.000000


## 2. Basic data cleaning

The next steps will reduce unnecessary columns, review missing data, and remove duplicate records.

### 5. Remove unused columns

We will remove selected amenity and listing-status columns that are not needed for the current cleaning process, then preview the remaining data.

In [5]:
df.drop(columns=['24x7 Security', "Children's Play Area", "Clubhouse", 'Intercom', 'Landscaped Gardens', 'Indoor Games', 'Gas Connection', "Jogging Track", "New/Resale"], inplace=True)
display(df.head())

,price,sqrt,location,bhk,Gymnasium,Lift Available,CarParking,Maintenance Staff,Swimming Pool
0,NaN,1498.0,Kharghar,3.0,1,No,1,0,1
1,125000000.0,2904.0,WORLI,4.0,1,1,1,1,1
2,43000000.0,2195.0,Goregaon East,3.0,1,1,1,No,1
3,4500000.0,738.0,Taloja,1.0,1,1,1,No,1
4,6000000.0,650.0,Mira Road East,2.0,0,1,0,NaN,0


### 6. Count rows with missing values

We will find how many records contain at least one missing value so we can plan how to handle incomplete data.

In [6]:
df[df.isna().any(axis=1)].shape

(3471, 9)

### 7. Check for completely empty rows

We will check whether any record is missing every value across the remaining columns.

In [7]:
df[df.isna().all(axis=1)].shape

(0, 9)

### 8. Examine the missing-value pattern

We will count how many missing values occur in each row to identify records with substantial gaps.

In [8]:
df.isna().sum(axis=1).value_counts()

0    3126
1    2422
2     836
3     186
4      26
5       1
Name: count, dtype: int64

### 9. Check for duplicate records

We will count duplicate rows before removing them from the dataset.

In [9]:
df.duplicated().sum()

np.int64(265)

### 10. Remove duplicate records

We will remove repeated rows and check the updated shape of the dataset.

In [10]:
df.drop_duplicates(inplace=True)
df.shape

(6332, 9)

### 11. Recheck numeric summary statistics

We will review the numeric columns again after duplicate removal to confirm the data still looks reasonable.

In [11]:
df.describe()

,price,sqrt,bhk
count,5.827000e+03,5826.000000,5824.000000
mean,1.508565e+07,1009.179540,1.906422
std,1.973683e+07,563.701681,0.866496
min,2.000000e+06,200.000000,1.000000
25%,5.274500e+06,650.000000,1.000000
50%,9.500000e+06,910.000000,2.000000
75%,1.750000e+07,1185.000000,2.000000
max,4.000000e+08,8511.000000,7.000000


### 12. Inspect location values

We will count each location, including missing entries, to understand the location data and spot inconsistent values.

In [12]:
df["location"].value_counts(dropna=False)

location
NaN                    508
Kharghar               488
Thane West             391
Mira Road East         357
Ulwe                   290
                      ... 
Tilak Nagar mumbai       1
Mahatma Gandhi Road      1
Pandurangwadi            1
gokuldham                1
Dokali Pada              1
Name: count, Length: 400, dtype: int64

### 13. Remove rows with too many missing values

We will keep only rows with three or fewer missing values, because rows with more gaps provide too little information for reliable cleaning.

In [13]:
df = df.drop(df[df.isna().sum(axis=1) > 3].index)
display(df.shape)

(6305, 9)

## 3. Clean the price column

The following steps will fill missing prices where a reliable group median is available, then remove any records that still cannot be assigned a price.

### 14. Fill prices using location and BHK groups

We will replace missing prices with the median price of properties that have the same location and number of bedrooms (BHK).

In [14]:
def getdata(df):

    group = df.groupby(['location', 'bhk'])

    for key, data in group:

        median_price = data['price'].median()
        
        null_rows = data[data['price'].isna()].index

        df.loc[null_rows, 'price'] = median_price

    return df

df = getdata(df)

### 15. Check the remaining missing prices

We will count how many price values are still missing after the location-and-BHK median fill.

In [15]:
df['price'].isna().sum()

np.int64(104)

### 16. Inspect records whose price is still missing

We will display key columns for the remaining records so we can see why their prices could not be filled.

In [16]:
remaining = df[df['price'].isna()]

display(
    remaining[['location', 'bhk', 'sqrt', 'price']].head(100)
)

,location,bhk,sqrt,price
36,Gundavali Gaothan,2.0,1100.0,NaN
86,matunga east,1.0,610.0,NaN
179,Malad West,NaN,350.0,NaN
204,Bhandup West,NaN,2000.0,NaN
207,Mira Road East,NaN,950.0,NaN
...,...,...,...,...
6021,NaN,5.0,8511.0,NaN
6144,Kurla East,NaN,750.0,NaN
6159,NaN,2.0,503.0,NaN
6218,Sector 20 Kharghar,NaN,530.0,NaN


### 17. Remove incomplete records that cannot support price filling

We will remove rows with a missing price when either the BHK or area (sqrt) value is also missing.

In [17]:
bad_rows = (df["price"].isna() & (df['bhk'].isna() | df['sqrt'].isna()))

df = df[~bad_rows]

### 18. Recheck missing prices after removing incomplete rows

We will count the remaining missing prices before applying a broader location-based estimate.

In [18]:
df['price'].isna().sum()

np.int64(64)

### 19. Fill remaining prices using location medians

We will replace any remaining missing prices with the median price for the same location.

In [19]:
def fill_by_location(df):

    group = df.groupby('location')

    for location, data in group:

        median_price = data['price'].median()

        null_rows = data[data['price'].isna()].index

        df.loc[null_rows, 'price'] = median_price

    return df

df = fill_by_location(df)

### 20. Confirm the result of location-based filling

We will check whether any price values are still missing after using the location median.

In [20]:
df['price'].isna().sum()

np.int64(48)

### 21. Remove any final rows without a price

We will drop any records that still have no price, leaving a dataset ready for analysis.

In [21]:
df.dropna(subset=['price'], inplace=True)
df['price'].isna().sum()

np.int64(0)

In [22]:
df.shape

(6217, 9)

In [23]:
print("Missing locations:", df['location'].isna().sum())
print("Unique locations:", df['location'].nunique())

Missing locations: 454
Unique locations: 388


In [24]:
df['location'] = df['location'].str.strip().str.lower()

In [25]:
for location in df['location'].unique():
    print(location)

kharghar
worli
goregaon east
taloja
mira road east
boisar
mulund east
ulwe
thane west
beturkar pada
koper khairane
kalyan east
panvel
kanjurmarg
juhu
sector 19 nerul
malad west
nan
powai
kandivali west
lower parel
seawoods
andheri  east
kurla
malad east
nala sopara
virar
andheri  west
mulund west
badlapur
jogeshwari west
virar west
mumbai
nerul
kalyan west
sector 19 kharghar
kalamboli
mira road and beyond
kamothe
mumbai agra national highway
dahisar west
dadar west
goregaon west
ghatkopar west
badlapur east
kandivali east
dahisar
borivali west
napeansea road
matunga east
ghansoli
sector 5
taloja panchanand
thane
sector 18
chembur
magathane
andheri
dombivali
bandra west
kalwa
dombivali east
ville parle east
thakur complex
sector 17 ulwe
naigaon east
bhayandar east
santacruz east
sainath nagar
dombivli (west)
sector 5 ulwe
sector 36 kharghar
ghatkopar east
sector 20 kharghar
kolshet road
mumbai highway
titwala
thakur village
wadala east wadala
khalapur
wadala
greater khanda
rajendra naga

In [26]:
print("Total rows:", len(df))
print("Missing locations:", df['location'].isna().sum())
print(
    "Missing %:",
    df['location'].isna().mean() * 100
)

Total rows: 6217
Missing locations: 454
Missing %: 7.302557503619108


In [27]:
missing_location = df[df['location'].isna()]

display(
    missing_location[
        ['price', 'sqrt', 'bhk']
    ].head(50)
)

,price,sqrt,bhk
19,23000000.0,1100.0,3.0
41,19500000.0,1215.0,3.0
45,30000000.0,2355.0,4.0
52,9500000.0,950.0,2.0
59,2390000.0,NaN,1.0
60,8100000.0,1097.0,2.0
67,8420000.0,960.0,2.0
78,19700000.0,709.0,2.0
93,17200000.0,750.0,2.0
95,4500000.0,550.0,1.0


In [28]:
df['location'] = df['location'].fillna('unknown')
df['location'].isna().sum()

np.int64(0)

In [29]:
print('total rows :',len(df))
print("Missing BHK:", df['bhk'].isna().sum())
print('missing percentage :',df['bhk'].isna().mean()*100)

print("\nBHK values:")
print(df['bhk'].value_counts(dropna=False).sort_index())

total rows : 6217
Missing BHK: 460
missing percentage : 7.39906707415152

BHK values:
bhk
1.0    2119
2.0    2330
3.0    1090
4.0     173
5.0      37
6.0       6
7.0       2
NaN     460
Name: count, dtype: int64


In [30]:
display(
    df[df['bhk'].isna()][
        ['price', 'location', 'sqrt', 'bhk']
    ].head(50)
)

df['bhk'].isna().sum()

,price,location,sqrt,bhk
12,4800000.0,kalyan east,855.0,NaN
28,2900000.0,taloja,695.0,NaN
35,25000000.0,andheri west,1250.0,NaN
47,15200000.0,nerul,1286.0,NaN
48,7500000.0,kharghar,1050.0,NaN
62,8500000.0,dahisar west,560.0,NaN
108,18000000.0,thane west,1309.0,NaN
111,16500000.0,borivali west,756.0,NaN
129,3645000.0,virar,810.0,NaN
145,18900000.0,ville parle east,600.0,NaN


np.int64(460)

In [31]:
def fill_bhk(row):

    if pd.isna(row['bhk']):

        if row['sqrt'] < 800:
            return 1
        elif row['sqrt'] < 1200:
            return 2
        elif row['sqrt'] < 1600:
            return 3
        else:
            return 4

    return row['bhk']

df['bhk'] = df.apply(fill_bhk, axis=1)


print('missing bhk:',df['bhk'].isna().sum())
print('missing sqrt:',df['sqrt'].isna().sum())

missing bhk: 0
missing sqrt: 487


In [32]:
df.head(50)

,price,sqrt,location,bhk,Gymnasium,Lift Available,CarParking,Maintenance Staff,Swimming Pool
0,14950000.0,1498.0,kharghar,3.0,1,No,1,0,1
1,125000000.0,2904.0,worli,4.0,1,1,1,1,1
2,43000000.0,2195.0,goregaon east,3.0,1,1,1,No,1
3,4500000.0,738.0,taloja,1.0,1,1,1,No,1
4,6000000.0,650.0,mira road east,2.0,0,1,0,NaN,0
5,3000000.0,1160.0,boisar,3.0,NaN,1,0,No,1
6,11500000.0,650.0,mulund east,1.0,1,1,0,0,NaN
7,5500000.0,700.0,ulwe,2.0,0,0,0,No,No
8,9700000.0,645.0,mira road east,3.0,NaN,1,1,No,NaN
9,10800000.0,928.0,thane west,2.0,1,1,0,0,1


In [33]:
def fill_sqrt(row):

    if pd.isna(row['sqrt']):

        if row['bhk'] == 1:
            return np.random.randint(500, 800)
        elif row['bhk'] == 2:
            return np.random.randint(801, 1200)
        elif row['bhk'] == 3:
            return np.random.randint(1201, 1600)
        elif row['bhk'] == 4:
            return np.random.randint(1601, 2500)

    return row['sqrt']

df['sqrt'] = df.apply(fill_sqrt, axis=1)

print('missing sqrt:',df['sqrt'].isna().sum())
df = df.dropna(subset=['sqrt'])
print('missing sqrt:',df['sqrt'].isna().sum())


missing sqrt: 1
missing sqrt: 0


In [34]:
df.describe()

,price,sqrt,bhk
count,6.216000e+03,6216.000000,6216.000000
mean,1.510721e+07,1012.098456,1.927767
std,1.963159e+07,550.258288,0.888522
min,2.000000e+06,200.000000,1.000000
25%,5.300000e+06,650.000000,1.000000
50%,9.500000e+06,910.000000,2.000000
75%,1.750000e+07,1192.250000,2.000000
max,4.000000e+08,7600.000000,7.000000


In [35]:
cols = [
    'Gymnasium',
    'Lift Available',
    'CarParking',
    'Maintenance Staff',
    'Swimming Pool'
]

df[cols] = df[cols].replace({
    'Yes': 1,
    'No': 0
})

In [36]:
amenity_cols = ['Gymnasium', 'Lift Available', 'CarParking', 'Maintenance Staff', 'Swimming Pool']
df[amenity_cols] = df[amenity_cols].fillna(0)

In [37]:
df.head(50)

,price,sqrt,location,bhk,Gymnasium,Lift Available,CarParking,Maintenance Staff,Swimming Pool
0,14950000.0,1498.0,kharghar,3.0,1,0,1,0,1
1,125000000.0,2904.0,worli,4.0,1,1,1,1,1
2,43000000.0,2195.0,goregaon east,3.0,1,1,1,0,1
3,4500000.0,738.0,taloja,1.0,1,1,1,0,1
4,6000000.0,650.0,mira road east,2.0,0,1,0,0,0
5,3000000.0,1160.0,boisar,3.0,0,1,0,0,1
6,11500000.0,650.0,mulund east,1.0,1,1,0,0,0
7,5500000.0,700.0,ulwe,2.0,0,0,0,0,0
8,9700000.0,645.0,mira road east,3.0,0,1,1,0,0
9,10800000.0,928.0,thane west,2.0,1,1,0,0,1


In [38]:
df.info()
df['Gymnasium'].value_counts(dropna=False)
df['Gymnasium'].map(type).value_counts()

<class 'pandas.DataFrame'>
Index: 6216 entries, 0 to 6596
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              6216 non-null   float64
 1   sqrt               6216 non-null   float64
 2   location           6216 non-null   str    
 3   bhk                6216 non-null   float64
 4   Gymnasium          6216 non-null   object 
 5   Lift Available     6216 non-null   object 
 6   CarParking         6216 non-null   object 
 7   Maintenance Staff  6216 non-null   object 
 8   Swimming Pool      6216 non-null   object 
dtypes: float64(3), object(5), str(1)
memory usage: 485.6+ KB


Gymnasium
<class 'str'>    4857
<class 'int'>    1359
Name: count, dtype: int64

In [39]:
columns = [
    'Gymnasium',
    'Lift Available',
    'CarParking',
    'Maintenance Staff',
    'Swimming Pool'
]

for col in columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [40]:
df.info()

<class 'pandas.DataFrame'>
Index: 6216 entries, 0 to 6596
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              6216 non-null   float64
 1   sqrt               6216 non-null   float64
 2   location           6216 non-null   str    
 3   bhk                6216 non-null   float64
 4   Gymnasium          6216 non-null   int64  
 5   Lift Available     6216 non-null   int64  
 6   CarParking         6216 non-null   int64  
 7   Maintenance Staff  6216 non-null   int64  
 8   Swimming Pool      6216 non-null   int64  
dtypes: float64(3), int64(5), str(1)
memory usage: 485.6 KB


In [41]:
df.isna().sum()

price                0
sqrt                 0
location             0
bhk                  0
Gymnasium            0
Lift Available       0
CarParking           0
Maintenance Staff    0
Swimming Pool        0
dtype: int64

In [42]:
df.duplicated().sum()

np.int64(70)

In [43]:
display(df[df.duplicated(keep=False)].sort_values(
    ['location', 'price']
))

,price,sqrt,location,bhk,Gymnasium,Lift Available,CarParking,Maintenance Staff,Swimming Pool
3039,3188000.0,797.0,ambernath west,1.0,0,1,0,0,0
3817,3188000.0,797.0,ambernath west,1.0,0,1,0,0,0
866,15000000.0,650.0,andheri west,1.0,0,0,0,0,0
2846,15000000.0,650.0,andheri west,1.0,0,0,0,0,0
593,8400000.0,515.0,borivali west,1.0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...
2680,8500000.0,1155.0,ulwe,2.0,0,0,0,0,0
2310,30000000.0,2000.0,vashi,4.0,0,1,0,0,0
2582,30000000.0,2000.0,vashi,4.0,0,1,0,0,0
940,32500000.0,1130.0,ville parle east,3.0,0,0,0,0,0


In [44]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df

,price,sqrt,location,bhk,Gymnasium,Lift Available,CarParking,Maintenance Staff,Swimming Pool
0,14950000.0,1498.0,kharghar,3.0,1,0,1,0,1
1,125000000.0,2904.0,worli,4.0,1,1,1,1,1
2,43000000.0,2195.0,goregaon east,3.0,1,1,1,0,1
3,4500000.0,738.0,taloja,1.0,1,1,1,0,1
4,6000000.0,650.0,mira road east,2.0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...
6141,11200000.0,725.0,chembur,1.0,1,1,1,1,0
6142,13500000.0,1391.0,thane west,3.0,1,1,1,0,1
6143,4800000.0,560.0,unknown,1.0,1,1,1,1,1
6144,18100000.0,762.0,thane west,2.0,1,0,0,0,1


In [45]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicates:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)

Shape: (6146, 9)

Missing values:
price                0
sqrt                 0
location             0
bhk                  0
Gymnasium            0
Lift Available       0
CarParking           0
Maintenance Staff    0
Swimming Pool        0
dtype: int64

Duplicates: 0

Data types:
price                float64
sqrt                 float64
location                 str
bhk                  float64
Gymnasium              int64
Lift Available         int64
CarParking             int64
Maintenance Staff      int64
Swimming Pool          int64
dtype: object


In [47]:
df.to_csv("cleaned_mumbai_house_data.csv", index=False)